# ローカル検証パイプライン on Google Colab — AI Agent Security

ハイブリッド運用の GPU 層を Colab で回す。**ランタイム → ハードウェアアクセラレータ → GPU** を選ぶこと。
- `gpt_oss`（gpt-oss-20b Q4, 約12GB）: 無料 **T4(16GB)** で可。
- `gemma_4`（gemma-4-26B Q4, 約16GB）: **L4(24GB) か A100** 推奨（Colab Pro / Pay-As-You-Go）。

手順: ①依存導入 → ②リポジトリ取得（SDK 同梱）→ ③SDK 確認（fallback）→ ④GGUF 取得 → ⑤実行。

## ① 依存（llama.cpp は CUDA ビルド）

In [ ]:
!pip -q install gymnasium 'pydantic>=2' huggingface_hub kaggle
# CUDA 版 llama-cpp-python（数分。事前ビルド wheel があればそれが入る）
import os
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
!pip -q install llama-cpp-python
import llama_cpp; print('llama_cpp OK')

## ② リポジトリ取得（clone で SDK も同梱）
本リポジトリを clone すると `validation/` と `vendor/aicomp_sdk_pkg/`（公式 SDK・MIT・git 同梱）が両方入る。
private repo なので clone には PAT / SSH が必要。`validation/` と `vendor/` を手元からアップロードしてもよい。

In [ ]:
# 本リポジトリを clone（vendor/aicomp_sdk_pkg/ = 公式 SDK も含まれる）
# !git clone https://<PAT>@github.com/<you>/kaggle_ai_agent_security.git repo
# %cd repo
# もしくは validation/ と vendor/ を手元からアップロード。
import os
assert os.path.isdir('validation'), 'リポジトリを clone するか validation/ を配置してください'
print('repo OK -> validation/', '+ vendor/' if os.path.isdir('vendor') else '(vendor 未配置 → ③ で補完)')

## ③ SDK 確認（clone 済みなら不要・fallback）
`vendor/aicomp_sdk_pkg/` が既にあれば何もしない。無い場合（validation/ だけ上げた等）のみ、
Kaggle API かアップロードしたコンペ zip から `vendor/aicomp_sdk_pkg/` へ展開する。

In [ ]:
import os, zipfile, glob
sdk = 'vendor/aicomp_sdk_pkg'
if os.path.isdir(os.path.join(sdk, 'aicomp_sdk')):
    print('SDK 同梱済み ->', sdk)
else:
    # fallback: コンペ zip から展開（Kaggle API かアップロード）
    # from google.colab import files; files.upload()  # kaggle.json を選択
    # !mkdir -p /root/.kaggle && cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    # !kaggle competitions download -c ai-agent-security-multi-step-tool-attacks -p .
    os.makedirs(sdk, exist_ok=True)
    z = glob.glob('vendor/*.zip') + glob.glob('*.zip')
    assert z, 'vendor/aicomp_sdk_pkg/ が無く、展開元の zip も見つかりません'
    with zipfile.ZipFile(z[0]) as f: f.extractall(sdk)
    print('extracted ->', sorted(os.listdir(sdk))[:5])

## ④ GGUF 取得（初回のみ）

In [ ]:
MODEL = 'gpt_oss'  # or 'gemma_4'（L4/A100 推奨）。gemma は Google ライセンス同意が必要な場合あり
!python -m validation.download_models {MODEL}

## ⑤ 実行（公開 LB 相関 = public、汎化代理 = provenance）

In [ ]:
ATTACK = 'baseline'  # or 'path/to/your/attack.py'
!python -m validation.run_validation \
    --attack {ATTACK} --agent {MODEL} \
    --guardrails public,provenance \
    --candidates 30 --budget-s 600 --env gym \
    --candidates-out runs/cand_{MODEL}.json --report-out runs/report_{MODEL}.txt